# Let's give our camera some depth

Shoutout to [Depth Anything V2](https://github.com/DepthAnything/Depth-Anything-V2) and [ExecuTorch's depth demo](https://executorch.ai/playground/webgpu/#/demo/depth) for the inspiration!

Try the included photo, upload your own or turn on your camera. Compare the original with a coloured depth map using the slider or side-by-side view. Everything runs in your browser.

Choose CPU or WebGPU with `DEVICE` and run the cells from top to bottom in a fresh kernel. The model predicts **relative depth**, not distances in metres.


In [ ]:
import base64
import json
import statistics
import time
from pathlib import Path

import numpy as np
import torch
import cv2
from PIL import Image, ImageOps
from IPython.display import display
from depth_anything_v2.dpt import DepthAnythingV2
from depth_display import colorize_depth

DEVICE = 'cpu'
torch.autograd.set_multithreading_enabled(False)
print('PyTorch:', torch.__version__, '| device:', DEVICE)


## 1. Load the pretrained model

We're using Depth Anything V2 Small with its original pretrained weights. The first run downloads them; later runs reuse the browser's cached copy.


In [ ]:
from depth_weights import load_checkpoint

model = DepthAnythingV2(encoder='vits', features=64,
                        out_channels=[48, 96, 192, 384]).eval()
checkpoint = await load_checkpoint()
model.load_state_dict(checkpoint, strict=True)
del checkpoint
for parameter in model.parameters():
    parameter.requires_grad_(False)
print(f'{sum(p.numel() for p in model.parameters()):,} parameters')


## 2. Pick a photo

Start with the included photo or change `IMAGE_PATH` to an image you've uploaded. The model takes a 224×224 RGB image.


In [ ]:
IMAGE_PATH = 'YellowLabradorLooking_new.jpg'
SIZE = 224
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

photo = ImageOps.exif_transpose(Image.open(IMAGE_PATH)).convert('RGB')
photo.thumbnail((640, 640), Image.Resampling.LANCZOS)
pixels = np.asarray(photo.resize((SIZE, SIZE), Image.Resampling.BILINEAR)).astype(np.float32) / 255
image = torch.from_numpy(((pixels - MEAN) / STD).transpose(2, 0, 1).copy()).unsqueeze(0)
display(photo, include=['image/png'])
print('Model input:', tuple(image.shape))


## 3. Check the starting prediction

Run the model on CPU once so we can check the compiled prediction against it.


In [ ]:
started = time.perf_counter()
with torch.inference_mode():
    expected = model(image)
eager_seconds = time.perf_counter() - started
assert expected.shape == (1, SIZE, SIZE)
assert torch.isfinite(expected).all()
print(f'Eager reference ready in {eager_seconds:.3f} s')


## 4. Compile for the selected device

The first call includes compilation. Later calls reuse the compiled model, keeping its weights on the selected device.


In [ ]:
if DEVICE == 'webgpu':
    from iree.runtime.webgpu import get_webgpu_device
    from iree.turbine.runtime.torch_device import TorchDevice
    upload_capacity = len(model.state_dict()) + 1
    gpu = TorchDevice(await get_webgpu_device(
        max_concurrent_operations=upload_capacity), name='webgpu')
    backend = gpu.backend
else:
    gpu = None
    backend = 'turbine_cpu'

model = model.to(DEVICE)
image_device = image.to(DEVICE)
if gpu is not None:
    await gpu.synchronize()
print('Device ready:', DEVICE)


In [ ]:
compiled_model = torch.compile(model, backend=backend, fullgraph=True, dynamic=False)
print('Compiling the model...', flush=True)
started = time.perf_counter()
with torch.inference_mode():
    depth = compiled_model(image_device)
if gpu is not None:
    await gpu.synchronize()
cold_seconds = time.perf_counter() - started
actual = await gpu.to_cpu(depth) if gpu is not None else depth.detach()
torch.testing.assert_close(actual, expected, rtol=3e-3, atol=1e-3)
print(f'Cold first call: {cold_seconds:.3f} s')
print(f'Max error against eager PyTorch: {(actual - expected).abs().max().item():.3g}')


## 5. See the depth map

The photo is on the left and its predicted depth is on the right. Colours are scaled for this image; they aren't a distance scale you can compare across photos.


In [ ]:
depth_array = actual[0].numpy()
assert np.isfinite(depth_array).all()
colours, display_range = colorize_depth(depth_array)
depth_image = Image.fromarray(colours).resize(photo.size, Image.Resampling.BILINEAR)
comparison = Image.new('RGB', (photo.width * 2, photo.height))
comparison.paste(photo, (0, 0))
comparison.paste(depth_image, (photo.width, 0))
display(comparison, include=['image/png'])
depth_image.save('predicted-depth.png')
np.save('relative-depth.npy', depth_array)
print('Saved predicted-depth.png and the raw relative-depth.npy array')


## 6. Time the model

Warm up, then measure seven completed predictions. Copying the result back is measured separately. Camera FPS below also includes preparing and displaying each frame.


In [ ]:
for _ in range(2):
    with torch.inference_mode():
        depth = compiled_model(image_device)
    if gpu is not None:
        await gpu.synchronize()

warm_times = []
for _ in range(7):
    started = time.perf_counter()
    with torch.inference_mode():
        depth = compiled_model(image_device)
    if gpu is not None:
        await gpu.synchronize()
    warm_times.append(time.perf_counter() - started)

started = time.perf_counter()
actual = await gpu.to_cpu(depth) if gpu is not None else depth.detach().clone()
readback_seconds = time.perf_counter() - started
torch.testing.assert_close(actual, expected, rtol=3e-3, atol=1e-3)
report = dict(model='Depth Anything V2 Small', device=DEVICE, precision='FP32',
              model_input=list(image.shape), cold_first_call_seconds=cold_seconds,
              completed_model_median_ms=statistics.median(warm_times) * 1000,
              output_copy_or_readback_ms=readback_seconds * 1000,
              max_abs_error=float((actual - expected).abs().max()), passed=True)
Path('depth-benchmark.json').write_text(json.dumps(report, indent=2))
print(f'Completed model call: {report["completed_model_median_ms"]:.1f} ms (median of 7)')
print(f'Output copy/readback: {report["output_copy_or_readback_ms"]:.1f} ms')
print(f'Max error against eager PyTorch: {report["max_abs_error"]:.3g}')
print('Full measurements saved in depth-benchmark.json')


## 7. Try your camera

Click **Start camera**, allow access and drag the slider to compare the original frame with its depth map. You can switch to side-by-side view or upload a photo too. **Stop camera** releases the camera and keeps the last result visible.

The panel shows completed updates per second. Colours rescale for each frame, so this shows relative depth, not measurements in metres.


In [ ]:
def prepare_frame(bgr):
    rgb = cv2.cvtColor(cv2.resize(bgr, (SIZE, SIZE), interpolation=cv2.INTER_LINEAR),
                       cv2.COLOR_BGR2RGB).astype(np.float32) / 255
    return torch.from_numpy(((rgb - MEAN) / STD).transpose(2, 0, 1).copy()).unsqueeze(0)


async def process_frame(jpeg):
    started = time.perf_counter()
    frame = cv2.imdecode(np.frombuffer(jpeg, dtype=np.uint8), cv2.IMREAD_COLOR)
    if frame is None:
        raise ValueError('Could not decode the camera frame.')
    model_input = prepare_frame(frame)
    prepared_at = time.perf_counter()
    device_input = model_input.to(DEVICE)
    with torch.inference_mode():
        prediction = compiled_model(device_input)
    prediction_cpu = await gpu.to_cpu(prediction) if gpu is not None else prediction.detach()
    model_at = time.perf_counter()
    colours, _ = colorize_depth(prediction_cpu[0].numpy())
    colours = cv2.resize(colours, (frame.shape[1], frame.shape[0]), interpolation=cv2.INTER_LINEAR)
    coloured_at = time.perf_counter()
    ok, encoded = cv2.imencode('.png', cv2.cvtColor(colours, cv2.COLOR_RGB2BGR),
                               [cv2.IMWRITE_PNG_COMPRESSION, 1])
    if not ok:
        raise RuntimeError('Could not encode the predicted depth image.')
    png = base64.b64encode(encoded.tobytes()).decode()
    ended_at = time.perf_counter()
    return dict(png=png, timings=dict(prepare_ms=1000 * (prepared_at - started),
        model_ms=1000 * (model_at - prepared_at), colour_ms=1000 * (coloured_at - model_at),
        encode_ms=1000 * (ended_at - coloured_at)))


from live_depth import LiveDepth

if 'camera' in globals():
    await camera.close()
camera = LiveDepth(process_frame, sample=IMAGE_PATH, size=SIZE, device=DEVICE)


To close the panel from Python, run `await camera.close()`. Re-running its cell closes the previous panel first.

Model and weights: [Depth Anything V2 Small](https://huggingface.co/depth-anything/Depth-Anything-V2-Small), Apache-2.0.

Photo: [YellowLabradorLooking_new.jpg](https://commons.wikimedia.org/wiki/File:YellowLabradorLooking_new.jpg), Elf; derivatives by Djmirko and FT2, CC BY-SA 3.0. Displayed derivatives retain that license. See `sources.json` and the included model license.
